# Chicken Muscle Instron Analysis
**Cyclic tensile loading — 2×2 preservation matrix**

Conditions: 5%/10% formalin × no glycerin/glycerin

**Cross-section note:** Filenames contain two dimension pairs `W1×H1-W2×H2` (mm):
- `W1×H1` = thick end cross-section; `W2×H2` = thin end cross-section
- E is computed with both areas (thick = lower bound, thin = upper bound, mean = nominal)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path
from scipy.signal import find_peaks
import re
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('.')  # run notebook from the data folder, or change this

CONDITION_COLORS = {
    '5% formalin / no glycerin':  '#1f77b4',
    '5% formalin / glycerin':     '#2ca02c',
    '10% formalin / no glycerin': '#d62728',
    '10% formalin / glycerin':    '#ff7f0e',
}
CONDITIONS = list(CONDITION_COLORS.keys())

MAX_CYCLE        = 100          # load up to this many cycles per trial
HIGHLIGHT_CYCLES = [1, 30, 60]  # snapshot cycles for overlays, energy & modulus charts

## 1. File discovery, condition parsing, and specimen dimensions

In [ ]:
# Load gauge lengths
lengths_df = pd.read_csv(DATA_DIR / 'specimen_lengths.csv').dropna(subset=['trial','length'])
lengths_df['trial']  = lengths_df['trial'].str.strip()
lengths_df['length'] = pd.to_numeric(lengths_df['length'], errors='coerce')
gauge_length_map = dict(zip(lengths_df['trial'], lengths_df['length']))
print('Gauge lengths loaded:')
for k, v in gauge_length_map.items():
    print(f'  {k[:65]} -> {v} mm')

In [ ]:
def parse_filename(path: Path):
    name = path.name
    formalin  = '10%' if name.startswith('10%') else '5%'
    glycerin  = 'glycerin' if re.search(r'with.?glyc', name, re.I) else 'no glycerin'
    condition = f'{formalin} formalin / {glycerin}'

    slip = ''
    if 'slippedatend' in name.lower():         slip = '[slipped at end]'
    elif 'slippedatbeginning' in name.lower():  slip = '[slipped at beginning]'

    dim_pairs = [(float(w), float(h)) for w, h in re.findall(r'([\d.]+)x([\d.]+)', name)]
    area_thick = area_thin = area_mean = None
    if len(dim_pairs) >= 2:
        a1, a2 = dim_pairs[0][0]*dim_pairs[0][1], dim_pairs[1][0]*dim_pairs[1][1]
        area_thick, area_thin = max(a1,a2), min(a1,a2)
        area_mean = (area_thick + area_thin) / 2
    elif len(dim_pairs) == 1:
        area_thick = area_thin = area_mean = dim_pairs[0][0] * dim_pairs[0][1]

    gauge_length = gauge_length_map.get(name, None)
    return condition, area_thick, area_thin, area_mean, gauge_length, slip


def load_csv(path: Path) -> pd.DataFrame:
    """Row layout: 0-5 preamble, 6 column names, 7 units, 8+ data."""
    df = pd.read_csv(path, skiprows=8, header=None,
                     names=['_idx','Time_s','Disp_mm','Force_kN'],
                     dtype=float, na_values=['','""'])
    df.drop(columns=['_idx'], inplace=True)
    df.dropna(inplace=True)
    df['Force_N'] = df['Force_kN'] * 1000.0
    df.reset_index(drop=True, inplace=True)
    return df


all_csvs = sorted(set(list(DATA_DIR.glob('*.csv')) + list(DATA_DIR.glob('*_1.csv'))))
all_csvs = [p for p in all_csvs
            if 'specimen_lengths' not in p.name and not p.name.startswith('results_')]

records = []
for p in all_csvs:
    cond, a_thick, a_thin, a_mean, g_len, slip = parse_filename(p)
    records.append(dict(path=p, condition=cond,
                        area_thick_mm2=a_thick, area_thin_mm2=a_thin, area_mean_mm2=a_mean,
                        gauge_length_mm=g_len, slip=slip))

meta = pd.DataFrame(records)
print(f'Found {len(meta)} trial CSVs\n')
display(meta[['path','condition','area_thick_mm2','area_thin_mm2','gauge_length_mm','slip']]
        .assign(path=meta['path'].apply(lambda p: p.name))
        .style.format({'area_thick_mm2':'{:.2f}','area_thin_mm2':'{:.2f}','gauge_length_mm':'{:.2f}'}))

## 2. Cycle segmentation & loading

In [ ]:
def segment_cycles(df: pd.DataFrame):
    disp = df['Disp_mm'].values
    dt   = np.median(np.diff(df['Time_s'].values))
    min_dist = max(20, int(2.0 / dt / 4))
    troughs, _ = find_peaks(-disp, distance=min_dist, prominence=0.05)
    return [df.iloc[troughs[i]: troughs[i+1]+1].copy() for i in range(len(troughs)-1)]


# ── Trials to exclude ────────────────────────────────────────────────────────
EXCLUDED_TRIALS = {
    '5%formalin_noglycerin_10.3x5.5-9.3x4.4[slippedatbeginning]_20260304_163205_1.csv',
    '5%formalin-noglycerin_11.7x6-8.44x3.1_[slippedatend]_20260304_170807_1.csv',
    '10%formalin-noglyc-11.3x5.25-8.4x4_20260304_153351.is_tcyclic.csv',
}
# ─────────────────────────────────────────────────────────────────────────────

all_data = []
for _, row in meta.iterrows():
    if row['path'].name in EXCLUDED_TRIALS:
        print(f"  [excluded] {row['path'].name[:70]}")
        continue
    try:
        df     = load_csv(row['path'])
        cycles = segment_cycles(df)
        all_data.append({**row.to_dict(), 'df': df, 'cycles': cycles, 'n_cycles': len(cycles)})
        print(f"  {row['path'].name[:65]}")
        print(f"    {row['condition']:38s}  cycles: {len(cycles):3d}  slip: '{row['slip']}'")
    except Exception as e:
        print(f"  !! {row['path'].name}: {e}")

print()
print(f"{'Condition':<38} {'File':<50} {'N cycles':>8}")
print('-'*98)
for t in all_data:
    print(f"{t['condition']:<38} {t['path'].name[:49]:<50} {t['n_cycles']:>8}")

## 3. Hysteresis plots — all trials, cycles 1–100

In [ ]:
all_disp, all_force = [], []
for trial in all_data:
    for cyc in trial['cycles'][:MAX_CYCLE]:
        all_disp.extend(cyc['Disp_mm'].tolist())
        all_force.extend(cyc['Force_N'].tolist())

pad_d = (max(all_disp) - min(all_disp)) * 0.05
pad_f = (max(all_force) - min(all_force)) * 0.05
XLIM = (min(all_disp)-pad_d, max(all_disp)+pad_d)
YLIM = (min(all_force)-pad_f, max(all_force)+pad_f)

linestyles = ['-','--','-.',':']
alphas = np.linspace(0.12, 0.90, MAX_CYCLE)

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
for ax, cond in zip(axes.flatten(), CONDITIONS):
    for t_idx, trial in enumerate([t for t in all_data if t['condition']==cond]):
        ls = linestyles[t_idx % len(linestyles)]
        slip_lbl = f" {trial['slip']}" if trial['slip'] else ''
        for c_idx, cyc in enumerate(trial['cycles'][:MAX_CYCLE]):
            lw    = 1.5 if c_idx in (0, 29, 59) else 0.5
            label = f"Trial {t_idx+1}{slip_lbl}" if c_idx==0 else None
            ax.plot(cyc['Disp_mm'], cyc['Force_N'],
                    color=CONDITION_COLORS[cond], alpha=alphas[c_idx],
                    linewidth=lw, linestyle=ls, label=label)
    ax.set_title(cond, fontsize=11, fontweight='bold')
    ax.set_xlim(XLIM); ax.set_ylim(YLIM)
    ax.set_xlabel('Displacement (mm)'); ax.set_ylabel('Force (N)')
    ax.legend(fontsize=7, loc='upper left'); ax.grid(True, alpha=0.3)

fig.suptitle('Hysteresis Curves — All Trials, Cycles 1–100\n'
             '(lighter = earlier, darker = later)', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('hysteresis_all_conditions.pdf', bbox_inches='tight', dpi=200)
plt.savefig('hysteresis_all_conditions.png', bbox_inches='tight', dpi=200)
plt.show()

## 4. Selected cycle overlay

In [ ]:
cycle_colors = {1: '#1a1aff', 30: '#ff8800', 60: '#cc0000'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
for ax, cond in zip(axes.flatten(), CONDITIONS):
    for t_idx, trial in enumerate([t for t in all_data if t['condition']==cond]):
        ls = linestyles[t_idx % len(linestyles)]
        slip_lbl = f" {trial['slip']}" if trial['slip'] else ''
        for cyc_num in HIGHLIGHT_CYCLES:
            idx = cyc_num - 1
            if idx >= len(trial['cycles']): continue
            ax.plot(trial['cycles'][idx]['Disp_mm'], trial['cycles'][idx]['Force_N'],
                    color=cycle_colors.get(cyc_num,'black'), linestyle=ls, linewidth=1.5,
                    label=f"T{t_idx+1}{slip_lbl} c{cyc_num}", alpha=0.85)
    ax.set_title(cond, fontsize=11, fontweight='bold')
    ax.set_xlim(XLIM); ax.set_ylim(YLIM)
    ax.set_xlabel('Displacement (mm)'); ax.set_ylabel('Force (N)')
    ax.legend(fontsize=6.5, loc='upper left', ncol=2); ax.grid(True, alpha=0.3)

legend_elements = [Line2D([0],[0], color=cycle_colors.get(c,'black'), lw=2, label=f'Cycle {c}')
                   for c in HIGHLIGHT_CYCLES]
fig.legend(handles=legend_elements, loc='upper right', fontsize=9,
           title='Cycle', bbox_to_anchor=(1.02, 1))
fig.suptitle(f'Selected Hysteresis Cycles {HIGHLIGHT_CYCLES}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('hysteresis_selected_cycles.pdf', bbox_inches='tight', dpi=200)
plt.savefig('hysteresis_selected_cycles.png', bbox_inches='tight', dpi=200)
plt.show()

## 5. Energy dissipation (hysteresis loop area)

Shoelace formula: $E_{\text{lost}} = \frac{1}{2}|\sum_i(D_i F_{i+1} - D_{i+1} F_i)|$ [N·mm = mJ]

In [ ]:
def hysteresis_area(cyc):
    D, F = cyc['Disp_mm'].values, cyc['Force_N'].values
    if len(D) < 3: return np.nan
    return 0.5 * abs(np.dot(D, np.roll(F,-1)) - np.dot(F, np.roll(D,-1)))


energy_rows = []
for trial in all_data:
    for c_idx, cyc in enumerate(trial['cycles'][:MAX_CYCLE]):
        energy_rows.append({'condition': trial['condition'],
                            'trial_file': trial['path'].name,
                            'slip': trial['slip'],
                            'cycle': c_idx + 1,
                            'energy_mJ': hysteresis_area(cyc)})
energy_df = pd.DataFrame(energy_rows)

summary_rows = []
for cond in CONDITIONS:
    sub = energy_df[energy_df['condition']==cond]
    for cyc_num in HIGHLIGHT_CYCLES:
        vals = sub[sub['cycle']==cyc_num]['energy_mJ'].dropna().values
        summary_rows.append({'Condition': cond, 'Cycle': cyc_num, 'N': len(vals),
                             'Mean (mJ)': np.mean(vals) if len(vals) else np.nan,
                             'Std (mJ)':  np.std(vals, ddof=1) if len(vals)>1 else np.nan})
summary_df = pd.DataFrame(summary_rows)
display(summary_df.style.format({'Mean (mJ)':'{:.4f}','Std (mJ)':'{:.4f}'})
        .set_caption('Energy dissipated per hysteresis loop (mJ)'))

### 5a. Energy vs. cycle number

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for cond in CONDITIONS:
    sub   = energy_df[energy_df['condition']==cond]
    means = sub.groupby('cycle')['energy_mJ'].mean()
    stds  = sub.groupby('cycle')['energy_mJ'].std(ddof=1).fillna(0)
    ax.plot(means.index, means.values, color=CONDITION_COLORS[cond], label=cond, lw=2)
    ax.fill_between(means.index, means-stds, means+stds,
                    color=CONDITION_COLORS[cond], alpha=0.15)
for c in HIGHLIGHT_CYCLES:
    ax.axvline(c, color='gray', linestyle=':', lw=0.8)
ax.set_xlabel('Cycle number'); ax.set_ylabel('Energy dissipated (mJ)')
ax.set_title('Hysteresis Energy Loss vs. Cycle Number (mean ± 1 SD)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('energy_vs_cycle.pdf', bbox_inches='tight', dpi=200)
plt.savefig('energy_vs_cycle.png', bbox_inches='tight', dpi=200)
plt.show()

### 5b. Energy at selected cycles — bar chart

In [ ]:
fig, axes = plt.subplots(1, len(HIGHLIGHT_CYCLES),
                         figsize=(5*len(HIGHLIGHT_CYCLES), 5), sharey=False)
for ax, cyc_num in zip(axes, HIGHLIGHT_CYCLES):
    sub = energy_df[energy_df['cycle']==cyc_num]
    means = [sub[sub['condition']==c]['energy_mJ'].mean() for c in CONDITIONS]
    stds  = [sub[sub['condition']==c]['energy_mJ'].std(ddof=1) for c in CONDITIONS]
    x = np.arange(len(CONDITIONS))
    ax.bar(x, [m if not np.isnan(m) else 0 for m in means],
           yerr=[s if not np.isnan(s) else 0 for s in stds],
           color=[CONDITION_COLORS[c] for c in CONDITIONS], capsize=6, alpha=0.85, ecolor='black')
    for i, cond in enumerate(CONDITIONS):
        vals = sub[sub['condition']==cond]['energy_mJ'].dropna().values
        ax.scatter(np.full(len(vals),i)+np.random.uniform(-0.1,0.1,len(vals)),
                   vals, color='black', s=25, zorder=5, alpha=0.7)
    ax.set_title(f'Cycle {cyc_num}', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(CONDITIONS, rotation=20, ha='right', fontsize=8)
    ax.set_ylabel('Energy dissipated (mJ)'); ax.grid(True, axis='y', alpha=0.3)
fig.suptitle(f'Energy per Hysteresis Loop — Cycles {HIGHLIGHT_CYCLES} (mean ± SD)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('energy_by_condition.pdf', bbox_inches='tight', dpi=200)
plt.savefig('energy_by_condition.png', bbox_inches='tight', dpi=200)
plt.show()

## 6. Young's modulus at selected cycles

$E = \sigma/\varepsilon$, fit to the middle 40% of each loading ramp.
- **E_thick**: lower bound (large area); **E_thin**: upper bound (small area); **E_nominal**: mean

In [ ]:
def youngs_modulus_from_cycle(cyc, gauge_length_mm, area_mm2, fit_fraction=0.4):
    if gauge_length_mm is None or area_mm2 is None: return {'E_MPa': np.nan, 'r2': np.nan}
    if np.isnan(gauge_length_mm) or np.isnan(area_mm2) or area_mm2==0:
        return {'E_MPa': np.nan, 'r2': np.nan}
    D, F = cyc['Disp_mm'].values, cyc['Force_N'].values
    peak_idx = int(np.argmax(D))
    D_load, F_load = D[:peak_idx+1], F[:peak_idx+1]
    if len(D_load) < 5: return {'E_MPa': np.nan, 'r2': np.nan}
    strain = D_load / gauge_length_mm
    stress = F_load / area_mm2
    s_lo = strain.min() + (strain.max()-strain.min()) * (0.5 - fit_fraction/2)
    s_hi = strain.min() + (strain.max()-strain.min()) * (0.5 + fit_fraction/2)
    mask = (strain >= s_lo) & (strain <= s_hi)
    if mask.sum() < 3: mask = np.ones(len(strain), dtype=bool)
    coeffs = np.polyfit(strain[mask], stress[mask], 1)
    E_MPa  = coeffs[0]
    pred   = np.polyval(coeffs, strain[mask])
    ss_res = np.sum((stress[mask]-pred)**2)
    ss_tot = np.sum((stress[mask]-stress[mask].mean())**2)
    r2 = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
    return {'E_MPa': E_MPa, 'r2': r2}


modulus_rows = []
for trial in all_data:
    for cyc_num in HIGHLIGHT_CYCLES:
        idx = cyc_num - 1
        if idx >= len(trial['cycles']): continue
        cyc = trial['cycles'][idx]
        g   = trial['gauge_length_mm']
        r_thick   = youngs_modulus_from_cycle(cyc, g, trial['area_thick_mm2'])
        r_thin    = youngs_modulus_from_cycle(cyc, g, trial['area_thin_mm2'])
        r_nominal = youngs_modulus_from_cycle(cyc, g, trial['area_mean_mm2'])
        modulus_rows.append({
            'condition':      trial['condition'],
            'trial_file':     trial['path'].name,
            'slip':           trial['slip'],
            'cycle':          cyc_num,
            'gauge_length_mm': g,
            'area_thick_mm2': trial['area_thick_mm2'],
            'area_thin_mm2':  trial['area_thin_mm2'],
            'E_thick_MPa':    r_thick['E_MPa'],
            'E_thin_MPa':     r_thin['E_MPa'],
            'E_nominal_MPa':  r_nominal['E_MPa'],
            'r2_nominal':     r_nominal['r2'],
        })
modulus_df = pd.DataFrame(modulus_rows)
print(f'modulus_df: {modulus_df.shape}  cycles: {sorted(modulus_df["cycle"].unique())}')

for cyc_num in HIGHLIGHT_CYCLES:
    sub = modulus_df[modulus_df['cycle']==cyc_num]
    print(f'\n── Cycle {cyc_num} ──')
    display(sub[['condition','trial_file','gauge_length_mm',
                 'E_thick_MPa','E_thin_MPa','E_nominal_MPa','r2_nominal']]
            .style.format({'gauge_length_mm':'{:.1f}','E_thick_MPa':'{:.3f}',
                           'E_thin_MPa':'{:.3f}','E_nominal_MPa':'{:.3f}','r2_nominal':'{:.4f}'}))

### 6a. Young's modulus — mean ± SD per condition

In [ ]:
E_summary_rows = []
for cond in CONDITIONS:
    for cyc_num in HIGHLIGHT_CYCLES:
        sub = modulus_df[(modulus_df['condition']==cond) & (modulus_df['cycle']==cyc_num)]
        for col, label in [('E_nominal_MPa','E_nominal'),('E_thick_MPa','E_thick'),('E_thin_MPa','E_thin')]:
            vals = sub[col].dropna().values
            E_summary_rows.append({'Condition': cond, 'Cycle': cyc_num, 'Estimate': label,
                                   'N': len(vals),
                                   'Mean (MPa)': np.mean(vals) if len(vals) else np.nan,
                                   'Std (MPa)':  np.std(vals,ddof=1) if len(vals)>1 else np.nan})
E_summary_df = pd.DataFrame(E_summary_rows)
display(E_summary_df[E_summary_df['Estimate']=='E_nominal'].drop(columns='Estimate')
        .style.format({'Mean (MPa)':'{:.3f}','Std (MPa)':'{:.3f}'})
        .set_caption("Young's modulus — nominal (mean cross-section area)"))

### 6b. Young's modulus bar chart

Bars = mean ± SD across trials. Gray whiskers = thick→thin area uncertainty per trial.

In [ ]:
fig, axes = plt.subplots(1, len(HIGHLIGHT_CYCLES),
                         figsize=(5*len(HIGHLIGHT_CYCLES), 5), sharey=True)
for ax, cyc_num in zip(axes, HIGHLIGHT_CYCLES):
    sub_all = modulus_df[modulus_df['cycle']==cyc_num]
    x = np.arange(len(CONDITIONS))
    for i, cond in enumerate(CONDITIONS):
        sub = sub_all[sub_all['condition']==cond]
        vals_nom   = sub['E_nominal_MPa'].dropna().values
        vals_thick = sub['E_thick_MPa'].dropna().values
        vals_thin  = sub['E_thin_MPa'].dropna().values
        mean_nom = np.mean(vals_nom) if len(vals_nom) else 0
        std_nom  = np.std(vals_nom, ddof=1) if len(vals_nom)>1 else 0
        ax.bar(i, mean_nom, yerr=std_nom,
               color=CONDITION_COLORS[cond], capsize=6, alpha=0.80, ecolor='black')
        jitter = np.random.uniform(-0.15, 0.15, len(vals_nom))
        ax.scatter(i+jitter, vals_nom, color='black', s=25, zorder=5, alpha=0.8)
        for j in range(min(len(vals_thick), len(vals_thin))):
            ax.plot([i+jitter[j], i+jitter[j]], [vals_thick[j], vals_thin[j]],
                    color='gray', lw=1.0, alpha=0.6, zorder=4)
    ax.set_title(f'Cycle {cyc_num}', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(CONDITIONS, rotation=18, ha='right', fontsize=8)
    ax.set_ylabel("Young's modulus (MPa)"); ax.grid(True, axis='y', alpha=0.3)
fig.suptitle(f"Young's Modulus at Cycles {HIGHLIGHT_CYCLES}\n"
             "(bars = mean ± SD; gray lines = thick→thin area uncertainty)",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('youngs_modulus.pdf', bbox_inches='tight', dpi=200)
plt.savefig('youngs_modulus.png', bbox_inches='tight', dpi=200)
plt.show()

### 6c. Young's modulus evolution (line plot)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
offsets = np.linspace(-1.5, 1.5, len(CONDITIONS))
for cond, offset in zip(CONDITIONS, offsets):
    means, stds, xs = [], [], []
    for cyc_num in HIGHLIGHT_CYCLES:
        sub  = modulus_df[(modulus_df['condition']==cond) & (modulus_df['cycle']==cyc_num)]
        vals = sub['E_nominal_MPa'].dropna().values
        if len(vals):
            means.append(np.mean(vals))
            stds.append(np.std(vals,ddof=1) if len(vals)>1 else 0)
            xs.append(cyc_num + offset)
    ax.errorbar(xs, means, yerr=stds, fmt='o-',
                color=CONDITION_COLORS[cond], label=cond, capsize=5, lw=2, markersize=7)
ax.set_xticks(HIGHLIGHT_CYCLES)
ax.set_xticklabels([f'Cycle {c}' for c in HIGHLIGHT_CYCLES])
ax.set_ylabel("Young's modulus (MPa)")
ax.set_title(f"Young's Modulus Evolution — Cycles {HIGHLIGHT_CYCLES} (mean ± SD)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('youngs_modulus_evolution.pdf', bbox_inches='tight', dpi=200)
plt.savefig('youngs_modulus_evolution.png', bbox_inches='tight', dpi=200)
plt.show()

## 7. Export results

In [ ]:
modulus_df.to_csv('results_youngs_modulus.csv', index=False)
energy_df.to_csv('results_energy_per_cycle.csv', index=False)
summary_df.to_csv('results_energy_summary.csv', index=False)
E_summary_df.to_csv('results_youngs_modulus_summary.csv', index=False)
for f in ['results_youngs_modulus.csv','results_energy_per_cycle.csv',
          'results_energy_summary.csv','results_youngs_modulus_summary.csv']:
    print(f'  {f}')